# NeuroBright: Results Analysis

Analyze model performance and visualize predictions.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from src.utils.signal_utils import load_config
from src.data.dataset import EEGDataset
from src.models.model_loader import ModelLoader

sns.set_style('whitegrid')
print("✓ Libraries imported")

In [ ]:
# Load configuration and model
config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_path = ModelLoader.get_latest_model(config)
model, checkpoint = ModelLoader.load_model(model_path, config, device)

print(f"Model loaded from: {model_path}")
print(f"Training metrics: {checkpoint['metrics']}")

In [ ]:
# Load dataset
dataset = EEGDataset.from_processed_dir(config['paths']['processed_data'])
train_dataset, val_dataset = dataset.split(
    val_ratio=config['training']['val_split'],
    random_seed=config['training']['random_seed']
)

print(f"Validation set size: {len(val_dataset)}")

In [ ]:
# Evaluate on validation set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for i in range(len(val_dataset)):
        window, label = val_dataset[i]
        window = window.unsqueeze(0).to(device)
        
        output = model(window)
        pred = output.argmax(dim=1).item()
        
        all_preds.append(pred)
        all_labels.append(label.item())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = (all_preds == all_labels).mean() * 100
print(f"Validation Accuracy: {accuracy:.2f}%")

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
class_names = config['model']['class_names']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Per-class accuracy
per_class_acc = []
for i, name in enumerate(class_names):
    mask = all_labels == i
    acc = (all_preds[mask] == all_labels[mask]).mean() * 100
    per_class_acc.append(acc)
    print(f"{name}: {acc:.2f}%")

# Plot
plt.figure(figsize=(8, 5))
plt.bar(class_names, per_class_acc, color=['#2ecc71', '#3498db', '#e74c3c'])
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim([0, 100])
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()